In [ ]:
import kagglehub
nikolasgegenava_sard_search_and_rescue_path = kagglehub.dataset_download('nikolasgegenava/sard-search-and-rescue')

print('Data source import complete.')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install ultralytics

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import random

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

torch.backends.cudnn.deterministic = True

In [ ]:
from ultralytics import YOLO

import random
import os

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() == True else 'cpu')
device

In [ ]:
from ultralytics import YOLO
import os


yaml_path = os.path.join(nikolasgegenava_sard_search_and_rescue_path, 'search-and-rescue', 'data.yaml')

model = YOLO('yolov5s.pt')

model.train(
    data=yaml_path,
    epochs=20,
    imgsz=640,
    batch=16,
    device=device
)

In [ ]:
import cv2

def normalize_image(image):
    return image / 255.0

def resize_image(image, size=(640, 640)):
    return cv2.resize(image, size)

dataset_path = '/kaggle/input/sard-search-and-rescue/search-and-rescue'
valid_images_path = os.path.join(dataset_path, 'test', 'images')

image_files = [file for file in os.listdir(valid_images_path) if file.endswith('.jpg')]

if len(image_files) > 0:
    num_images = len(image_files)
    step_size = max(1, num_images // 9)
    selected_images = [image_files[i] for i in range(0, num_images, step_size)]

    fig, axes = plt.subplots(3, 3, figsize=(20, 21))
    fig.suptitle('Validation Set Inferences', fontsize=24)

    for i, ax in enumerate(axes.flatten()):
        if i < len(selected_images):
            image_path = os.path.join(valid_images_path, selected_images[i])

            image = cv2.imread(image_path)

            if image is not None:
                resized_image = resize_image(image, size=(640, 640))
                normalized_image = normalize_image(resized_image)

                normalized_image_uint8 = (normalized_image * 255).astype(np.uint8)

                results = model.predict(source=normalized_image_uint8, imgsz=640, conf=0.5)

                annotated_image = results[0].plot(line_width=1)
                annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
                ax.imshow(annotated_image_rgb)
            else:
                print(f"Failed to load image {image_path}")
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
#!zip -r /kaggle/working/runs.zip /kaggle/working/runs

In [ ]:
from google.colab import drive
import shutil
import os

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir rutas
ruta_origen = '/content/runs/detect/train/weights/best.pt'

carpeta_destino = '/content/drive/MyDrive/TFG'
ruta_destino_final = os.path.join(carpeta_destino, 'best.pt')

# 3. Copiar
if os.path.exists(ruta_origen):
    if not os.path.exists(carpeta_destino):
        print(f" No se encuentra la carpeta '{carpeta_destino}'. creando.")
        os.makedirs(carpeta_destino)

    shutil.copy(ruta_origen, ruta_destino_final)
    print(f" modelo guardado en: {ruta_destino_final}")

    shutil.copy(ruta_origen, '/content/best.pt')
    print(" Copia temporal lista para la App.")
else:
    print(f"Error: No se encuentra el archivo en {ruta_origen}")